# 第5章 回归模型 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]
- [[课程/深度学习/PPT/Chapter2/Chapter2.md]]（仅误差与泛化差距口径）
- [[课程/深度学习/演示代码/Chapter5.py]]
- [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md]]
- [[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md]]
- [[课程/深度学习/作业/Homework03/Homework03.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q5B 30分钟综合题：带噪传感器校准与最佳快照

**新编迁移题，依据** [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#五、显式正则化]]、[[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#六、频谱偏置与早停]]，不同于A卷的无噪声五维响应场。

**共100分，30分钟，从空白独立编程。**

1. **数据（20分）**：CPU单线程，torch.manual_seed(52)。实现`target_b5(x)=sin(2*x[:,0])+.3*x[:,0]`。顺序生成128/64/96条一维均匀[-1,1]训练/验证/测试输入，命名Xb_train/Xb_val/Xb_test；训练标签yb_train加.12*randn噪声，yb_val/yb_test为干净目标。
2. **模型与目标（20分）**：net_b5为1→24→1 Tanh网络；SGD(.05)。实现`l2_b5(net)`，含权重及偏置的平方和。每步优化数据MSE+.001*L2，不能用weight_decay重复惩罚。
3. **训练与早停（30分）**：至多120步；每步更新后重新评估train data MSE、objective、validation MSE，依次存history_b5字典列表，键data_mse/objective/val_mse。验证MSE严格降低即clone保存最佳state_dict与1起始best_epoch_b5，否则未改善计数+1；连续12次未改善停。预算结束也要恢复最佳模型。**测试集不参与决定停止**。
4. **评估（15分）**：恢复后输出report_b5的train_mse/train_rmse/test_rmse/gap_abs，以及best_val_b5。允许未触发早停；记录len(history_b5)为实际更新数。训练误差相对有噪声训练标签计算。
5. **图与解释（15分）**：保存output/Q5B.png，横轴Update，画更新后训练data MSE与validation MSE两线，标记best epoch；不能混画objective冒充训练MSE。写一句为何验证用于早停而测试用于最终报告。

注意：参数快照必须独立于后续训练；本卷不做庞大调参。


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

torch.manual_seed(52)
def target_b5(x):
    return torch.sin(2*x[:,0])+.3*x[:,0]
def l2_b5(net):
    return sum(p.square().sum() for p in net.parameters())
Xb_train = 2*torch.rand(128,1)-1
Xb_val = 2*torch.rand(64,1)-1
Xb_test = 2*torch.rand(96,1)-1
yb_train = target_b5(Xb_train)+.12*torch.randn(128)
yb_val, yb_test = target_b5(Xb_val), target_b5(Xb_test)
net_b5 = nn.Sequential(nn.Linear(1,24), nn.Tanh(), nn.Linear(24,1))
opt_b5 = torch.optim.SGD(net_b5.parameters(), lr=.05, weight_decay=0)
best_val_b5, waiting_b5, best_state_b5 = float("inf"), 0, None
history_b5 = []
for epoch in range(1,121):
    net_b5.train()
    opt_b5.zero_grad()
    mse = (net_b5(Xb_train).squeeze(-1)-yb_train).square().mean()
    objective = mse+.001*l2_b5(net_b5)
    objective.backward()
    opt_b5.step()
    net_b5.eval()
    with torch.no_grad():
        data_mse = (net_b5(Xb_train).squeeze(-1)-yb_train).square().mean().item()
        val_mse = (net_b5(Xb_val).squeeze(-1)-yb_val).square().mean().item()
        obj = data_mse+.001*l2_b5(net_b5).item()
    history_b5.append(dict(data_mse=data_mse, objective=obj, val_mse=val_mse))
    if val_mse < best_val_b5:
        best_val_b5, best_epoch_b5, waiting_b5 = val_mse, epoch, 0
        best_state_b5 = {k:v.detach().clone() for k,v in net_b5.state_dict().items()}
    else:
        waiting_b5 += 1
    if waiting_b5 >= 12:
        break
net_b5.load_state_dict(best_state_b5)
with torch.no_grad():
    train_mse_b5 = (net_b5(Xb_train).squeeze(-1)-yb_train).square().mean().item()
    test_rmse_b5 = (net_b5(Xb_test).squeeze(-1)-yb_test).square().mean().sqrt().item()
train_rmse_b5 = float(np.sqrt(train_mse_b5))
report_b5 = dict(train_mse=train_mse_b5, train_rmse=train_rmse_b5,
                 test_rmse=test_rmse_b5, gap_abs=abs(test_rmse_b5-train_rmse_b5))
Path("output").mkdir(exist_ok=True)
fig, ax = plt.subplots()
ax.plot(range(1,len(history_b5)+1), [r["data_mse"] for r in history_b5], label="Train data MSE")
ax.plot(range(1,len(history_b5)+1), [r["val_mse"] for r in history_b5], label="Validation MSE")
ax.axvline(best_epoch_b5, ls="--", color="gray", label="Best epoch")
ax.set(xlabel="Update", ylabel="MSE", title="Noisy sensor calibration")
ax.legend()
fig.tight_layout()
fig.savefig("output/Q5B.png", dpi=120)
plt.close(fig)
print("Best epoch / total:", best_epoch_b5, len(history_b5), "Best val:", best_val_b5)
print(report_b5)
print("验证集选择停止和快照，测试集不参与选择，最后再报告。")


### 解析与点评

本题把L2、验证早停、快照恢复放进同一个完整任务；参数数24×2+25=73。监控验证只需前向，不需反传。保存的历史为更新后的数据MSE，与objective分开。噪声训练RMSE可能高于干净测试RMSE，所以课程gap要取绝对值，不能预设差的符号。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
assert Xb_train.shape == (128,1) and Xb_val.shape == (64,1) and Xb_test.shape == (96,1)
assert torch.allclose(target_b5(torch.zeros(1,1)), torch.zeros(1))
assert net_b5(torch.zeros(1,1)).shape == (1,1)
assert sum(p.numel() for p in net_b5.parameters()) == 73
assert 1 <= best_epoch_b5 <= len(history_b5) <= 120
assert all(r["objective"] >= r["data_mse"] for r in history_b5)
assert np.isclose(best_val_b5, min(r["val_mse"] for r in history_b5))
with torch.no_grad():
    restored_b5 = (net_b5(Xb_val).squeeze(-1)-yb_val).square().mean().item()
assert np.isclose(restored_b5, best_val_b5, rtol=1e-5, atol=1e-8)
assert np.isfinite(list(report_b5.values())).all()
assert np.isclose(report_b5["gap_abs"], abs(report_b5["test_rmse"]-report_b5["train_rmse"]))
assert Path("output/Q5B.png").stat().st_size > 1000

print("自检通过")
